A script implementing BICePs to reweight populations for a simple three state
toy model system.  Here, our prior comes from random generation of the Boltzmann
distribution and reweighting is performed using two experimental observables both set to 0.0 A.U.

For more details about this toy model systema and visual aids, please refer to
this notebook: `examples/enforcing_uniform_reference.ipynb`

In [28]:
import sys, os
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import biceps
from biceps.PosteriorSampler import u_kln_and_states_kn
from pymbar import MBAR

In [29]:
class Data:
    def __init__(self, array_list):
        self.array_list = array_list

    def save(self, filename):
        with open(filename, 'wb') as f:
            pickle.dump(self.array_list, f)

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            array_list = pickle.load(f)
        return cls(array_list)

In [30]:
def write_noe_files(weights, x, exp, dir):
    for i in range(len(weights)):
        model = pd.read_pickle("template.noe")
        _model = pd.DataFrame()

        for j in range(len(exp)):
            row = model.iloc[0].copy()
            row["restraint_index"] = int(exp[j][0])
            row["atom_index1"] = int(exp[j][1])
            row["atom_index2"] = int(exp[j][2])
            row["exp"] = float(exp[j][3])
            row["model"] = float(x[i][j])  # x[i] must be flat and match len(exp)
            _model = pd.concat([_model, row.to_frame().T], ignore_index=True)

        _model.to_pickle(f"{dir}/{i}.noe")


###### Parameters #######

In [31]:
nStates,Nd = 76,1 # 76 distance with 1 NOE observables
n_xis,n_lambdas,nreplicas,nsteps,change_Nr_every,swap_every=1,2,1,1000000,0,0
multiprocess=4
σ_prior=0.161 # 0.08, 0.16
stat_model,data_uncertainty="Bayesian","single"
data_likelihood = "gaussian" #"log normal" # "gaussian"

write_every = 10
attempt_move_state_every = 1
attempt_move_sigma_every = 1

Make output directories

In [32]:
state_dir = f"{nStates}_state"
biceps.toolbox.mkdir(state_dir)

datapoints_dir = f"{state_dir}/{nStates}_state_{Nd}_datapoints"
biceps.toolbox.mkdir(datapoints_dir)

dir = f"{datapoints_dir}/Prior_error_{σ_prior}"
biceps.toolbox.mkdir(dir)

## Loaded the population

In [33]:
clustering = pd.read_csv(f"../clustering/cluster_percentages.csv")

populations = np.array(clustering["Population"])
populations.shape

(92,)

In [34]:
energies = -np.log(populations)
energies.shape

(92,)

## Load the Prior Model (From MD) Calculated NOE distances 

In [35]:
md_distances = pd.read_csv(f"../clustering/md_distances.csv")

forward_model_data = np.array(md_distances)
forward_model_data.shape

(92, 76)

## Load the Refer NMR (Experimental Measurement)

In [36]:
restraints_table = {
    'weak': 5,
    'medium': 3.5,
    'strong': 2.5
}

dist_res_file = '../../../../utils/nspe_7_1_restraints.csv'
df_dist_res = pd.read_csv(dist_res_file)

for i, row in df_dist_res.iterrows():
    df_dist_res.at[i, df_dist_res.columns[2]] = restraints_table[row[2]]  # Correct mapping

# Make a look up table for intensity 
df_dist_res
dist_res = df_dist_res.values.tolist()
dist_res[:5]

[[160, 165, 5], [160, 166, 5], [161, 165, 5], [161, 166, 5], [137, 165, 5]]

In [37]:
np.shape(dist_res)

(76, 3)

In [38]:
### Extract the restraints_index from the labels 

human_readable_labels = "../clustering/human_readable_labels.csv"
df_labels = pd.read_csv(human_readable_labels)
df_labels[:5]
restraint_index = pd.factorize(df_labels['0'])[0] + 1

In [39]:
## Create experiment input with a list of [restraint_index, atom_index1, atom_index2, exp]

exp = [[r] + d for r, d in zip(restraint_index, dist_res)]
exp[:5]

type(exp)

list

## Write the NOE files 

In [40]:
data_dir = f"{dir}/NOE"
biceps.toolbox.mkdir(data_dir)

write_noe_files(weights=energies, x=forward_model_data, exp=exp, dir=data_dir)

In [41]:
df = pd.read_pickle(f"{data_dir}/1.noe")

df[:7]

,restraint_index,atom_index1,res1,atom_name1,atom_index2,res2,atom_name2,exp,model,comments
0,1,160,UNK1,H1,165,UNK1,H20,5.0,3.590082,NaN
1,1,160,UNK1,H1,166,UNK1,H20,5.0,3.526355,NaN
2,1,161,UNK1,H1,165,UNK1,H20,5.0,3.535191,NaN
3,1,161,UNK1,H1,166,UNK1,H20,5.0,3.519043,NaN
4,2,137,UNK1,H1,165,UNK1,H20,5.0,3.642625,NaN
5,2,137,UNK1,H1,166,UNK1,H20,5.0,3.613061,NaN
6,2,138,UNK1,H1,165,UNK1,H20,5.0,4.585789,NaN


## Load the input data

In [42]:
input_data = biceps.toolbox.sort_data(data_dir)
print(f"Input data: {biceps.toolbox.list_extensions(input_data)}")
forward_model_data = np.array([pd.read_pickle(i)["model"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/*.noe")])
experiment = np.array([pd.read_pickle(i)["exp"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/0.noe")])[0]


Input data: ['.noe']


In [43]:
outdir = f'{dir}/{stat_model}_{data_uncertainty}_sigma/{nsteps}_steps_{nreplicas}_replicas_{n_lambdas}_lam__swap_every_{swap_every}'
biceps.toolbox.mkdir(outdir)
print(f"nSteps of sampling: {nsteps}\nnReplicas: {nreplicas}")
lambda_values = np.linspace(0.0, 1.0, n_lambdas)

nSteps of sampling: 1000000
nReplicas: 1


In [44]:
sigMin,sigMax,dsig = 0.001,200,1.02
arr = np.exp(np.arange(np.log(sigMin), np.log(sigMax), np.log(dsig)))
l = len(arr)
sigma_index = round(l*0.73)

In [45]:
beta,beta_index=(1., 2.0, 1),0
_arr = np.linspace(*beta)
_l = len(_arr)
print("Alpha starts here: ",_arr[beta_index])
phi,phi_index=(1., 2.0, 1),0
gamma,gamma_index=(1.0, 2.0, np.e),0

Alpha starts here:  1.0


In [46]:
options = [dict(ref="uniform", stat_model=stat_model,
            sigma=(sigMin, sigMax, dsig), sigma_index=sigma_index, gamma=gamma,
            beta=beta, beta_index=beta_index, phi=phi, phi_index=phi_index,
            data_uncertainty=data_uncertainty, data_likelihood=data_likelihood,
            )]
print(pd.DataFrame(options))


       ref stat_model               sigma  sigma_index  \
0  uniform   Bayesian  (0.001, 200, 1.02)          450   

                           gamma           beta  beta_index            phi  \
0  (1.0, 2.0, 2.718281828459045)  (1.0, 2.0, 1)           0  (1.0, 2.0, 1)   

   phi_index data_uncertainty data_likelihood  
0          0           single        gaussian  


In [47]:
ensemble = biceps.ExpandedEnsemble(lambda_values=lambda_values, energies=energies)
ensemble.initialize_restraints(input_data, options, verbose=1)
print("ensemble.expanded_values = ",ensemble.expanded_values)

Time to initalize restraints: 0.20s
ensemble.expanded_values =  [(0.0, 1.0), (1.0, 1.0)]


In [48]:
sampler = biceps.PosteriorSampler(ensemble, nreplicas, change_Nr_every, write_every=write_every)
sampler.sample(nsteps, attempt_lambda_swap_every=swap_every, swap_sigmas=1,
        attempt_move_state_every=attempt_move_state_every,
        attempt_move_sigma_every=attempt_move_sigma_every,
        verbose=0, progress=1, multiprocess=True, capture_stdout=0)

 ██████████████████████████████▏ 100.0% [1000000/1000000 | 88.3 kHz | 1 | 0s | 11s] MCMC 


In [49]:
expanded_values = sampler.expanded_values
A = biceps.Analysis(sampler, outdir=outdir, nstates=len(energies), MBAR=True, multiprocess=False, capture_stdout=0)
A.plot(plottype="step", figsize=(12,14), figname=f"BICePs.pdf", pad=0.35, plot_all_distributions=1)
plt.show()
A.plot_energy_trace()
plt.show()
BS, pops = A.f_df, A.P_dP[:,len(expanded_values[:])-1]
BS /= sampler.nreplicas
K = len(expanded_values[:])-1
pops_std = A.P_dP[:,2*K]
print(f"Predicted populatins: {pops}")

These states have not been sampled:
 [12 13 40 50 56 66 73 78]
These states have not been sampled:
 [11 17 24 26 30 35 36 38 40 42 44 46 47 49 50 51 53 55 56 57 60 61 63 64
 66 67 68 69 70 71 73 76 77 78 82 83 86 87 90 91]
 ██████████████████████████████▏ 100.0% [100000/100000 | 406.9 kHz | 1 | 0s | 0s] u_kln  
Time for MBAR: 3.150 s
Writing 76_state/76_state_1_datapoints/Prior_error_0.161/Bayesian_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/BS.dat...
Writing 76_state/76_state_1_datapoints/Prior_error_0.161/Bayesian_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/populations.dat...
Top 9 states: [22, 6, 19, 15, 74, 10, 0, 7, 9]
Top 9 populations: [0.00192973 0.00333053 0.00352009 0.00479211 0.00679245 0.01759529
 0.18491341 0.21412807 0.55375591]
nplots =  1


/var/folders/d8/y2dvs1ln1gjcwccrkvtffr240000gn/T/ipykernel_60448/3964779525.py:4: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Predicted populatins: [1.84913409e-01 1.38186373e-03 4.68292496e-06 3.89295863e-05
 1.61586055e-05 5.14207047e-06 3.33052610e-03 2.14128066e-01
 6.01163047e-05 5.53755913e-01 1.75952922e-02 1.96732655e-08
 3.40553259e-06 3.80322452e-05 4.55701143e-04 4.79210955e-03
 5.62346407e-05 4.99139971e-07 1.58052624e-04 3.52008663e-03
 1.68622759e-05 1.15291625e-05 1.92973063e-03 9.87742052e-04
 2.04856455e-06 7.49480842e-06 1.44019825e-06 2.99044110e-04
 2.61533617e-06 3.07136444e-04 1.95833695e-07 8.21340395e-06
 5.01416548e-06 1.46697438e-04 5.10285026e-04 1.87625566e-06
 2.29220796e-07 2.73547165e-04 1.90969017e-07 3.00176793e-05
 0.00000000e+00 5.76973869e-04 9.92087973e-08 2.99934851e-04
 3.68692394e-07 9.85870051e-05 3.92692756e-08 1.31370477e-07
 4.30436644e-05 4.45161575e-06 0.00000000e+00 1.12171265e-05
 3.73335084e-05 3.05749608e-07 1.50734121e-05 9.77804961e-06
 0.00000000e+00 9.05493416e-07 2.43891080e-05 9.79747293e-06
 3.44564593e-08 1.83873622e-06 4.69723042e-06 1.01329570e-07
 5

In [50]:
pops.shape

# Convert to DataFrame
df_predicted_population = pd.DataFrame(pops)

# Save to CSV
df_predicted_population.to_csv("../clustering/predicted_population_biceps.csv", index=False)

In [51]:
most_populated_index = np.argmax(pops)
most_populated_value = pops[most_populated_index]

print(f"Most populated state: {most_populated_index} with value {most_populated_value}")


Most populated state: 9 with value 0.5537559130353115


In [52]:
top5_indices = np.argsort(pops)[-5:][::-1]  # Sort, take last 5, reverse for descending order
top5_values = pops[top5_indices]

for i, (idx, val) in enumerate(zip(top5_indices, top5_values), 1):
    print(f"Top {i}: index = {idx}, population = {val:.4f}")

top5_indices

Top 1: index = 9, population = 0.5538
Top 2: index = 7, population = 0.2141
Top 3: index = 0, population = 0.1849
Top 4: index = 10, population = 0.0176
Top 5: index = 74, population = 0.0068


array([ 9,  7,  0, 10, 74])

In [53]:
BS, pops = A.f_df, A.P_dP
pops0 = pops[:, 0]  # Prior
pops1 = pops[:, len(sampler.lambda_values) - 1]  # Posterior

fig = A.plot(
    plottype="step",
    figsize=(6, 7),
    figname="BICePs.pdf",
    pad=0.35,
    plot_all_distributions=1
)
axes = fig.get_axes()
ax_pop = axes[0]

# Annotate selected states
for i in range(len(pops1)):
    if (i == 0) or (pops1[i] > 0.0009):
        ax_pop.text(pops0[i], pops1[i], str(i), color='g')
        
fig.savefig(f"{outdir}/BICePs_labeled.pdf", dpi=300, bbox_inches='tight')

Top 9 states: [22, 6, 19, 15, 74, 10, 0, 7, 9]
Top 9 populations: [0.00192973 0.00333053 0.00352009 0.00479211 0.00679245 0.01759529
 0.18491341 0.21412807 0.55375591]
nplots =  1
